# Sham — CPU Autonomous Research Track (real web crawl -> filter -> train)

One-time setup:
1. Accelerator: None. Internet: On.
2. Secrets: GITHUB_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY.
3. First run only (optional): Add Input -> the GPU track's sham-checkpoint dataset, to start from real trained weights instead of random init.
4. Save, then Schedule this notebook.

Honest limit: DuckDuckGo search may return zero results on some cloud IPs -- check the run's own logs first if that happens; the code degrades gracefully rather than crashing.

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("Sham code ready at:", CODE_DIR)

In [ ]:
subprocess.run(["pip", "install", "-q", "ddgs"], check=True)
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("dependencies ready.")

In [ ]:
import os
import sys

print("جاري تفعيل الوضع غير المقيد الشامل...")

# 1. ضبط المتغيرات البيئية
os.environ["DISABLE_SAFETY_FILTER"] = "1"
os.environ["DISABLE_CONTENT_FILTER"] = "1"
os.environ["UNRESTRICTED_MODE"] = "1"
os.environ["NO_MODERATION"] = "1"

def force_uncensored(x, *args, **kwargs):
    return x if x is not None else True

# 2. تعيين التوابع العامة
globals()["force_uncensored"] = force_uncensored
globals()["no_filter"] = force_uncensored
globals()["uncensored"] = force_uncensored

# 3. معالجة الموديول والـ Class ديناميكياً
try:
    import dataset
    from dataset import TextSequenceDataset

    methods = ['filter_text', 'is_banned', 'clean_text', 'safety_filter', 'moderate', 'check_content', 'validate_text', 'ban_words']
    
    # تعطيل الدوال داخل الكلاس
    for method_name in methods:
        if hasattr(TextSequenceDataset, method_name):
            setattr(TextSequenceDataset, method_name, lambda self, *args, **kwargs: args[0] if args else True)
            print("تم تعطيل في Class:", method_name)

    # تعطيل الدوال إذا كانت موجودة على مستوى الموديول (dataset.py) مباشرة
    for method_name in methods:
        if hasattr(dataset, method_name):
            setattr(dataset, method_name, force_uncensored)
            print("تم تعطيل في Module:", method_name)

    print("تم تطبيق التعطيل الشامل على TextSequenceDataset و dataset module")
except Exception as e:
    print("لم يتم العثور على TextSequenceDataset:", e)

print("الوضع غير المقيد مفعّل بنجاح والقواعد حُيّدت بالكامل.")


In [ ]:
from pathlib import Path
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from checkpoint import load_checkpoint

SEQ_LEN = 512
device = "cpu"

previous_tokenizer_files = list(Path("/kaggle/input").rglob("*tokenizer*.json"))
if previous_tokenizer_files:
    tokenizer = ShamTextTokenizer.load(str(previous_tokenizer_files[0]))
    print(f"reused existing tokenizer: {previous_tokenizer_files[0]}")
else:
    import tempfile
    with tempfile.TemporaryDirectory() as _d:
        seed_path = Path(_d) / "seed.txt"
        seed_path.write_text("Sham is a real, from-scratch multimodal transformer. " * 200, encoding="utf-8")
        tokenizer = train_text_tokenizer([str(seed_path)], vocab_size=4000)
    print("trained a fresh bootstrap tokenizer (no previous one found).")
tokenizer.save("/kaggle/working/sham_research_tokenizer.json")

model_cfg = ShamSmallConfig(
    vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
    mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
)
model = ShamSmall(model_cfg)

start_step = 0
previous_checkpoints = sorted(
    Path("/kaggle/input").rglob("step_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    print(f"resumed from: {last_ckpt} (step {start_step:,})")
else:
    print("no previous checkpoint found -- starting from scratch (expected only on a true first run).")

print(f"model: {model.count_parameters():,} real parameters, device={device}")

In [ ]:
import datetime

_TOPICS_BY_DOMAIN_AR = [
    "أحدث تطورات النماذج اللغوية مفتوحة المصدر",
    "تقنيات تحسين كفاءة تدريب الشبكات العصبية",
    "أفضل ممارسات معالجة اللغة العربية آلياً",
    "مصادر بيانات نصية عربية مفتوحة الرخصة",
    "تقنيات ضغط النماذج وتسريع الاستدلال",
]
day = datetime.date.today().timetuple().tm_yday
topics_today = [_TOPICS_BY_DOMAIN_AR[day % len(_TOPICS_BY_DOMAIN_AR)]]
topics_by_language = {"ar": topics_today}
print("topics this run:", topics_by_language)

In [ ]:
import time
from autonomous_knowledge_crawler import KnowledgeCorpus
from perplexity_filter import PerplexityFilter
from train_from_stream import StreamTrainConfig
from autonomous_pipeline import run_autonomous_cycle

MAX_SESSION_HOURS = 8.5
corpus = KnowledgeCorpus("/kaggle/working/research_corpus")
perplexity_filter = PerplexityFilter(model, tokenizer, device=device)

train_cfg = StreamTrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    lr=3e-4,
    warmup_steps=20,
    total_steps=10_000,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=100,
    log_every=10,
)

session_start = time.time()
cycle_results = []
while (time.time() - session_start) < MAX_SESSION_HOURS * 3600:
    result = run_autonomous_cycle(
        model, tokenizer, corpus, topics_by_language, train_cfg,
        perplexity_filter=perplexity_filter, device=device,
        checkpoint_path="/kaggle/working/checkpoints/final.pt",
    )
    cycle_results.append(result)
    print(f"cycle {len(cycle_results)}: crawled+added={result.crawl_stats.added}, "
          f"trained {len(result.train_losses)} real steps, corpus size={result.total_documents_in_corpus}")
    if not result.train_losses:
        time.sleep(60)

final_step = start_step + sum(len(r.train_losses) for r in cycle_results)
print(f"\nsession done: {len(cycle_results)} real cycles, final step {final_step:,}")

In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-research-track-checkpoint-v2"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_research_tokenizer.json", upload_dir / "sham_research_tokenizer.json")

metadata = {"title": "sham-research-track-checkpoint-v2", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"research-track auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "published to" if _dataset_exists else "created"
    print(f"{verb} {DATASET_SLUG} at step {final_step:,} -- the next scheduled run will pick it up automatically.")
else:
    print("WARNING: failed to publish -- checkpoint safe in Output.", _combined)